In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#FEAT
from pathlib import Path
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from data_loader_clean import *
# Historical configuration omitted; provide values through private configuration.
from GPN_V5_adaMulti_clean_FEAT import *
import higher
from draw_single_model_pic import *
from thop import profile, clever_format
from para_and_indexes_collection_v1 import FewShotDeploymentProfiler
#约定：所有训练时使用类数<configured>抽<configured>，总样本<configured>个，每个任务中样本数支持集<configured>，查询集<configured>，
#     所有测试时使用类数<configured>抽<configured>-<configured>，总样本<configured>个，每个任务中样本数支持集<configured>,<configured>,<configured>，查询集<configured>，

# 参数设置
N_WAY = _cfg_require('GPN_V5.ipynb.cell0.module.N_WAY')
K_SHOT = _cfg_require('GPN_V5.ipynb.cell0.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V5.ipynb.cell0.module.Q_QUERY')
NUM_TASKS = _cfg_require('GPN_V5.ipynb.cell0.module.NUM_TASKS')
EPOCHS = _cfg_require('GPN_V5.ipynb.cell0.module.EPOCHS')
BLR = _cfg_require('GPN_V5.ipynb.cell0.module.BLR')
BATCH_SIZE =_cfg_require('GPN_V5.ipynb.cell0.module.BATCH_SIZE')
SEED = _cfg_require('GPN_V5.ipynb.cell0.module.SEED')#<configured>
SAVE_MODEL_DIR = Path(_cfg_require('GPN_V5.ipynb.cell0.module.path'))
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SAVE_MODEL_PATH=SAVE_MODEL_DIR/_cfg_require('GPN_V5.ipynb.cell0.module.path__2')
LOAD_MODEL_PATH=_cfg_require('GPN_V5.ipynb.cell0.module.LOAD_MODEL_PATH')
# Historical configuration omitted; provide values through private configuration.
dataset_path = _cfg_require('GPN_V5.ipynb.cell0.module.dataset_path')


def main():
    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('GPN_V5.ipynb.cell0.main.train_ratio'),max_sample_count=_cfg_require('GPN_V5.ipynb.cell0.main.max_sample_count'),
                            )
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V5.ipynb.cell0.main.train_ratio__2'),max_sample_count=_cfg_require('GPN_V5.ipynb.cell0.main.max_sample_count__2'),
                            )

    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V5.ipynb.cell0.main.num_tasks'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('GPN_V5.ipynb.cell0.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V5.ipynb.cell0.main.batch_size'), shuffle=True)

    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = GPN_Optimized(
                 reduction=_cfg_require('GPN_V5.ipynb.cell0.main.reduction'))

    params = count_parameters(model)
    
    
    x=torch.randn(2,1,_cfg_require('GPN_V5.ipynb.cell0.main.size_or_budget'),_cfg_require('GPN_V5.ipynb.cell0.main.size_or_budget__2'))
    # 前向传播
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values are positive: {(s > 0).all().item()}")

    # 带中间特征输出
    v, s, features = model(x, return_intermediate=True)
    print(f"\nIntermediate features:")
    for name, feat in features.items():
        if isinstance(feat, torch.Tensor):
            print(f"  {name}: {feat.shape}")
    input_tensor = torch.randn(1, 1, _cfg_require('GPN_V5.ipynb.cell0.main.size_or_budget__3'), _cfg_require('GPN_V5.ipynb.cell0.main.size_or_budget__4'))
    
    flops, params = profile(model, inputs=(input_tensor,))
    flops, params = clever_format([flops, params], "%.3f")
    print(f"FLOPs: {flops}, Params: {params}")
    # 创建训练器
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V5.ipynb.cell0.main.use_Mdistance'),use_multi=_cfg_require('GPN_V5.ipynb.cell0.main.use_multi'),save_middle=_cfg_require('GPN_V5.ipynb.cell0.main.save_middle'),
                              crop_ratio_h=_cfg_require('GPN_V5.ipynb.cell0.main.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V5.ipynb.cell0.main.crop_ratio_l'),resize=_cfg_require('GPN_V5.ipynb.cell0.main.resize'),FEAT_heads=_cfg_require('GPN_V5.ipynb.cell0.main.FEAT_heads'))
                              # Historical configuration omitted; provide values through private configuration.
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,epochs=EPOCHS,n_ways=N_WAY,save_path=SAVE_MODEL_PATH)
    print("训练完成！")
    
def inference():
    '\n    推理示例：加载已保存的模型进行预测\n    \n    !!!若报模型结构错误，检查：<configured>.引用是否为对应模型\n    '
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,
                            train_ratio=_cfg_require('GPN_V5.ipynb.cell0.inference.train_ratio'),max_sample_count=_cfg_require('GPN_V5.ipynb.cell0.inference.max_sample_count'),
                            )
    model = GPN_Optimized(
                 reduction=_cfg_require('GPN_V5.ipynb.cell0.inference.reduction'),
                 )
    
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V5.ipynb.cell0.inference.use_Mdistance'),use_multi=_cfg_require('GPN_V5.ipynb.cell0.inference.use_multi'),save_middle=_cfg_require('GPN_V5.ipynb.cell0.inference.save_middle'),
                              crop_ratio_h=_cfg_require('GPN_V5.ipynb.cell0.inference.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V5.ipynb.cell0.inference.crop_ratio_l'),resize=_cfg_require('GPN_V5.ipynb.cell0.inference.resize'),FEAT_heads=_cfg_require('GPN_V5.ipynb.cell0.inference.FEAT_heads'))
                              # Historical configuration omitted; provide values through private configuration.
    meta_learner.load_model(LOAD_MODEL_PATH)
    
    profiler = FewShotDeploymentProfiler(
        trainer=meta_learner,
        device=device,
        input_size=_cfg_require('GPN_V5.ipynb.cell0.inference.input_size'),
    )

    report_feat = profiler.generate_profile_report(
        test_dataset=test_dataset,
        n_way=_cfg_require('GPN_V5.ipynb.cell0.inference.n_way'),
        k_shot=_cfg_require('GPN_V5.ipynb.cell0.inference.k_shot'),
        q_query=_cfg_require('GPN_V5.ipynb.cell0.inference.q_query'),
        n_warmup=_cfg_require('GPN_V5.ipynb.cell0.inference.n_warmup'),
        n_trials=_cfg_require('GPN_V5.ipynb.cell0.inference.n_trials'),
        num_workers=_cfg_require('GPN_V5.ipynb.cell0.inference.num_workers'),
    )
    PROFILE_DIR = Path(_cfg_require('GPN_V5.ipynb.cell0.inference.path'))
    PROFILE_DIR.mkdir(parents=True, exist_ok=True)

    # 保存 JSON，保留完整字段
    with open(PROFILE_DIR / "report_feat.json", "w", encoding="utf-8") as f:
        json.dump(report_feat, f, indent=4, ensure_ascii=False)

    # 保存 CSV，方便论文表格整理
    pd.DataFrame([report_feat]).to_csv(
        PROFILE_DIR / "report_feat.csv",
        index=False,
        encoding="utf-8-sig"
    )
    # 评估
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    print("推理完成")
if __name__ == '__main__':
    # main()
    inference()
    
# % 时间计算：单任务构建+预测时间
# Historical configuration omitted; provide values through private configuration.
# % hierarchical：<configured>3ms
# % pairwise：<configured>5ms
# % dbscan：<configured>9ms
# % IMP：<configured>8ms
# %均采用warm-up策略去除第一个任务

In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
# UNEM
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from data_loader_clean import UAVDataset, MetaDataset
from GPN_V5_adaMulti_clean_UNEM import GPN_Optimized, GPNTrainer
from thop import profile, clever_format
from draw_single_model_pic import plot_all_visualizations
from para_and_indexes_collection_v1 import FewShotDeploymentProfiler
import json

# -------------------- 配置参数 --------------------
N_WAY = _cfg_require('GPN_V5.ipynb.cell1.module.N_WAY')
K_SHOT = _cfg_require('GPN_V5.ipynb.cell1.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V5.ipynb.cell1.module.Q_QUERY')
NUM_TASKS = _cfg_require('GPN_V5.ipynb.cell1.module.NUM_TASKS')
EPOCHS = _cfg_require('GPN_V5.ipynb.cell1.module.EPOCHS')
BLR = _cfg_require('GPN_V5.ipynb.cell1.module.BLR')
BATCH_SIZE = _cfg_require('GPN_V5.ipynb.cell1.module.BATCH_SIZE')
SEED = _cfg_require('GPN_V5.ipynb.cell1.module.SEED')

SAVE_MODEL_DIR = Path(_cfg_require('GPN_V5.ipynb.cell1.module.path'))
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SAVE_MODEL_PATH = SAVE_MODEL_DIR / _cfg_require('GPN_V5.ipynb.cell1.module.path__2')
LOAD_MODEL_PATH = SAVE_MODEL_PATH

dataset_path = _cfg_require('GPN_V5.ipynb.cell1.module.dataset_path')

# -------------------- 主函数 --------------------
def main():
    # 构建训练集/测试集
    train_dataset = UAVDataset(
        data_dir_path=dataset_path,
        seed=SEED,
        is_train=True,
        train_ratio=_cfg_require('GPN_V5.ipynb.cell1.main.train_ratio'),
        max_sample_count=_cfg_require('GPN_V5.ipynb.cell1.main.max_sample_count')
    )
    test_dataset = UAVDataset(
        data_dir_path=dataset_path,
        seed=SEED,
        is_train=False,
        train_ratio=_cfg_require('GPN_V5.ipynb.cell1.main.train_ratio__2'),
        max_sample_count=_cfg_require('GPN_V5.ipynb.cell1.main.max_sample_count__2')
    )

    meta_train_dataset = MetaDataset(train_dataset, 8, 15, 5, num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY, num_tasks=_cfg_require('GPN_V5.ipynb.cell1.main.num_tasks'))

    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('GPN_V5.ipynb.cell1.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V5.ipynb.cell1.main.batch_size'), shuffle=True)

    # 创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GPN_Optimized(reduction=_cfg_require('GPN_V5.ipynb.cell1.main.reduction'))

    # 测试前向传播
    x = torch.randn(2,1,_cfg_require('GPN_V5.ipynb.cell1.main.size_or_budget'),_cfg_require('GPN_V5.ipynb.cell1.main.size_or_budget__2'))
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values positive: {(s > 0).all().item()}")

    # FLOPs 统计
    flops, params = profile(model, inputs=(torch.randn(1,1,_cfg_require('GPN_V5.ipynb.cell1.main.size_or_budget__3'),_cfg_require('GPN_V5.ipynb.cell1.main.size_or_budget__4')),))
    flops, params = clever_format([flops, params], "%.3f")
    print(f"FLOPs: {flops}, Params: {params}")

    # 创建训练器，启用 UNEM
    # 创建训练器：UNEM-Class-GMM
    meta_learner = GPNTrainer(
        model,
        device=device,
        mlr=BLR,

        use_multi=_cfg_require('GPN_V5.ipynb.cell1.main.use_multi'),
        use_unem=_cfg_require('GPN_V5.ipynb.cell1.main.use_unem'),
        unem_gmm_components=None,      # None = 每类 k_shot 个 Gaussian component
        unem_layers=_cfg_require('GPN_V5.ipynb.cell1.main.unem_layers'),
        unem_use_diag_precision=_cfg_require('GPN_V5.ipynb.cell1.main.unem_use_diag_precision'),

        use_Mdistance=_cfg_require('GPN_V5.ipynb.cell1.main.use_Mdistance'),
        save_middle=_cfg_require('GPN_V5.ipynb.cell1.main.save_middle'),
        crop_ratio_h=_cfg_require('GPN_V5.ipynb.cell1.main.crop_ratio_h'),
        crop_ratio_l=_cfg_require('GPN_V5.ipynb.cell1.main.crop_ratio_l'),
        resize=_cfg_require('GPN_V5.ipynb.cell1.main.resize'),
    )

    # 训练
    print("开始训练...")
    meta_learner.train(train_loader=train_loader, test_loader=test_loader, epochs=EPOCHS, n_ways=N_WAY, save_path=SAVE_MODEL_PATH)
    print("训练完成！")

# -------------------- 推理函数 --------------------
def inference():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 创建模型
    model = GPN_Optimized(reduction=_cfg_require('GPN_V5.ipynb.cell1.inference.reduction'))
    meta_learner = GPNTrainer(
        model,
        device=device,
        mlr=BLR,

        use_multi=_cfg_require('GPN_V5.ipynb.cell1.inference.use_multi'),
        use_unem=_cfg_require('GPN_V5.ipynb.cell1.inference.use_unem'),
        unem_gmm_components=None,
        unem_layers=_cfg_require('GPN_V5.ipynb.cell1.inference.unem_layers'),
        unem_use_diag_precision=_cfg_require('GPN_V5.ipynb.cell1.inference.unem_use_diag_precision'),

        use_Mdistance=_cfg_require('GPN_V5.ipynb.cell1.inference.use_Mdistance'),
        save_middle=_cfg_require('GPN_V5.ipynb.cell1.inference.save_middle'),
        crop_ratio_h=_cfg_require('GPN_V5.ipynb.cell1.inference.crop_ratio_h'),
        crop_ratio_l=_cfg_require('GPN_V5.ipynb.cell1.inference.crop_ratio_l'),
        resize=_cfg_require('GPN_V5.ipynb.cell1.inference.resize'),
    )

    # 加载模型
    meta_learner.load_model(LOAD_MODEL_PATH)

    # 构建测试集
    test_dataset = UAVDataset(
        data_dir_path=dataset_path,
        seed=SEED,
        is_train=False,
        train_ratio=_cfg_require('GPN_V5.ipynb.cell1.inference.train_ratio'),
        max_sample_count=_cfg_require('GPN_V5.ipynb.cell1.inference.max_sample_count')
    )

    # 评估
    results = meta_learner.full_evaluation(test_dataset, n_trials=_cfg_require('GPN_V5.ipynb.cell1.inference.n_trials'), q_query=_cfg_require('GPN_V5.ipynb.cell1.inference.q_query'))

    # 可视化
    plot_all_visualizations(results, output_dir=_cfg_require('GPN_V5.ipynb.cell1.inference.output_dir'))
    print("推理完成")

    
def profile_unem():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    test_dataset = UAVDataset(
        data_dir_path=dataset_path,
        seed=SEED,
        is_train=False,
        train_ratio=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.train_ratio'),
        max_sample_count=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.max_sample_count'),
    )

    model = GPN_Optimized(reduction=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.reduction'))

    meta_learner = GPNTrainer(
        model=model,
        device=device,
        mlr=BLR,

        # 关键：UNEM 走多原型入口
        use_multi=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.use_multi'),

        # 关键：开启 UNEM
        use_unem=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.use_unem'),

        # 如果是每类一个 Gaussian
        unem_layers=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.unem_layers'),
        unem_use_diag_precision=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.unem_use_diag_precision'),
        use_Mdistance=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.use_Mdistance'),
        save_middle=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.save_middle'),
        crop_ratio_h=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.crop_ratio_h'),
        crop_ratio_l=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.crop_ratio_l'),
        resize=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.resize'),
    )

    meta_learner.load_model(LOAD_MODEL_PATH)

    profiler = FewShotDeploymentProfiler(
        trainer=meta_learner,
        device=device,
        input_size=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.input_size'),
    )

    report_unem = profiler.generate_profile_report(
        test_dataset=test_dataset,
        n_way=N_WAY,
        k_shot=K_SHOT,
        q_query=Q_QUERY,
        n_warmup=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.n_warmup'),
        n_trials=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.n_trials'),
        num_workers=_cfg_require('GPN_V5.ipynb.cell1.profile_unem.num_workers'),
    )
    PROFILE_DIR = Path(_cfg_require('GPN_V5.ipynb.cell1.profile_unem.path'))
    PROFILE_DIR.mkdir(parents=True, exist_ok=True)

    # 保存 JSON，保留完整字段
    with open(PROFILE_DIR / "report_unem.json", "w", encoding="utf-8") as f:
        json.dump(report_unem, f, indent=4, ensure_ascii=False)

    # 保存 CSV，方便论文表格整理
    pd.DataFrame([report_unem]).to_csv(
        PROFILE_DIR / "report_unem.csv",
        index=False,
        encoding="utf-8-sig"
    )
    return report_unem

# -------------------- 执行 --------------------
if __name__ == '__main__':
    # main()
    # inference()
    report=profile_unem()